# Rider Accident Forecasting & Analysis (Q3-Q4 / 2569)
**วัตถุประสงค์:**
1. Clean ข้อมูลและจัดการ Format วันที่เพี้ยน (พ.ศ./ค.ศ., YYYY-MM-DD, DD/MM/YYYY)
2. จัดการเรื่อง Unnamed columns และ Column Mapping ระหว่างปี 2568 กับ 2569
3. พยากรณ์ (Predict) โอกาสการเกิดอุบัติเหตุตามประเภท Campaign / Cause Theme ในไตรมาส 3-4 ปี 2569 แยกตามพื้นที่
4. สรุปผลการพยากรณ์และแมพวิเคราะห์ตามกรอบ **4M1E**

---

แบบรวม cause (แยกแค่ theme)

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

## 1. Load Data & Clean Unnamed Columns

In [ ]:
# โหลดไฟล์ข้อมูล
df68 = pd.read_excel("../data/300968-2568-AccidentRider-ฟอร์ม(ทวนสอบ130169)(2).xlsx", sheet_name=5)
df69 = pd.read_excel("../data/2569-AccidentRider-ฟอร์ม5.xlsx", sheet_name=0)

# Clean Unnamed columns
df68 = df68.loc[:, ~df68.columns.str.contains("^Unnamed", na=False)].dropna(axis=1, how="all")
df69 = df69.loc[:, ~df69.columns.str.contains("^Unnamed", na=False)].dropna(axis=1, how="all")

# Align Column Names
rename_dict_69 = {
    "TradeArea จากร้านถึงจุดเกิดเหตุ": "Trade Area จากร้านถึงจุดเกิดเหตุ",
    "TradeArea ร้านถึงลูกค้ารับของ(กม.)": "Trade Area ร้านถึงลูกค้ารับของ (กม.)",
    "ผลของการเกิดอุบัติเหตุ (เสียชีวิต/สูญเสียอวัยวะ/บาดเจ็บ)": "(เสียชีวิต/สูญเสียอวัยวะ/บาดเจ็บ)"
}
df69 = df69.rename(columns=rename_dict_69)
df69 = df69.drop(columns=["หลักสูตรความปลอดภัย6ชม.(ไม่เก็บเงิน)"], errors="ignore")

print("df68 Shape:", df68.shape)
print("df69 Shape:", df69.shape)

## 2. Robust Date Standardizer & Quarter Extraction
แปลง พ.ศ. เป็น ค.ศ. แล้วใช้ `pd.to_datetime` จัดการ Format วันที่ที่เพี้ยน (ทั้ง YYYY-MM-DD และ DD/MM/YYYY) พร้อมดึง Year, Month, Quarter อัตโนมัติ

In [ ]:
def find_date_column(df):
    """ฟังก์ชันช่วยหาชื่อคอลัมน์วันที่ แม้จะมีช่องว่างหรือเคาะเว้นวรรคไม่เท่ากัน"""
    for col in df.columns:
        # ค้นหาคอลัมน์ที่มีคำว่า 'วันที่เกิดอุบัติเหตุ' หรือ 'พ.ศ.'
        if "วันที่เกิดอุบัติเหตุ" in str(col) or "พ.ศ." in str(col):
            return col
    return None

def process_buddhist_dates(df):
    # 1. ค้นหาชื่อคอลัมน์วันที่จริงใน DataFrame
    date_col = find_date_column(df)
    
    if date_col is None:
        print("⚠️ ไม่พบคอลัมน์วันที่ใน DataFrame โปรดตรวจสอบชื่อคอลัมน์อีกครั้ง")
        print("คอลัมน์ที่มีอยู่ได้แก่:", df.columns.tolist())
        return df

    # 2. แปลงเป็น String และลบเวลา 00:00:00 ออก
    s = df[date_col].astype(str).str.replace(r"\s+\d{2}:\d{2}:\d{2}.*", "", regex=True)
    
    # 3. แปลง พ.ศ. (25xx) เป็น ค.ศ. (20xx) โดยลบ 543
    s_gregorian = s.str.replace(
        r"25(\d{2})", 
        lambda m: str(int("25" + m.group(1)) - 543), 
        regex=True
    )
    
    # 4. ให้ Pandas Parse วันที่อัตโนมัติ
    dt_series = pd.to_datetime(s_gregorian, dayfirst=True, errors='coerce')
    
    # 5. ดึง Year (พ.ศ.), Month และ Quarter
    df["year"] = (dt_series.dt.year + 543).astype("Int64")
    df["month"] = dt_series.dt.month.astype("Int64")
    df["quarter"] = "Q" + dt_series.dt.quarter.astype(str)
    
    # เติม Fallback สำหรับแถวที่แปลงไม่ได้จริงๆ
    df["quarter"] = df["quarter"].replace("Qnan", np.nan)
    return df

# Apply เข้า Dataset ทั้งสองปีโดยไม่ต้องระบุชื่อคอลัมน์ตรงๆ
df68 = process_buddhist_dates(df68)
df69 = process_buddhist_dates(df69)

print("df68 Quarter Counts:\n", df68['quarter'].value_counts(dropna=False))
print("\ndf69 Quarter Counts:\n", df69['quarter'].value_counts(dropna=False))

In [ ]:
def extract_quarter_from_month(df):
    # 1. ค้นหาคอลัมน์ที่มีคำว่า 'เดือน'
    month_col = None
    for col in df.columns:
        if "เดือน" in str(col):
            month_col = col
            break
            
    if month_col is None:
        print("⚠️ ไม่พบคอลัมน์เดือนใน DataFrame")
        return df

    # Map ชื่อเดือน/ตัวเลขเดือน ทุกรูปแบบเข้ากับ Quarter
    month_map = {
        # Q1
        'ม.ค.': 'Q1', 'มกราคม': 'Q1', 'jan': 'Q1', 'january': 'Q1', '1': 'Q1', '01': 'Q1', '1.0': 'Q1',
        'ก.พ.': 'Q1', 'กุมภาพันธ์': 'Q1', 'feb': 'Q1', 'february': 'Q1', '2': 'Q1', '02': 'Q1', '2.0': 'Q1',
        'มี.ค.': 'Q1', 'มีนาคม': 'Q1', 'mar': 'Q1', 'march': 'Q1', '3': 'Q1', '03': 'Q1', '3.0': 'Q1',
        # Q2
        'เม.ย.': 'Q2', 'เมษายน': 'Q2', 'apr': 'Q2', 'april': 'Q2', '4': 'Q2', '04': 'Q2', '4.0': 'Q2',
        'พ.ค.': 'Q2', 'พฤษภาคม': 'Q2', 'may': 'Q2', '5': 'Q2', '05': 'Q2', '5.0': 'Q2',
        'มิ.ย.': 'Q2', 'มิถุนายน': 'Q2', 'jun': 'Q2', 'june': 'Q2', '6': 'Q2', '06': 'Q2', '6.0': 'Q2',
        # Q3
        'ก.ค.': 'Q3', 'กรกฎาคม': 'Q3', 'jul': 'Q3', 'july': 'Q3', '7': 'Q3', '07': 'Q3', '7.0': 'Q3',
        'ส.ค.': 'Q3', 'สิงหาคม': 'Q3', 'aug': 'Q3', 'august': 'Q3', '8': 'Q3', '08': 'Q3', '8.0': 'Q3',
        'ก.ย.': 'Q3', 'กันยายน': 'Q3', 'sep': 'Q3', 'september': 'Q3', '9': 'Q3', '09': 'Q3', '9.0': 'Q3',
        # Q4
        'ต.ค.': 'Q4', 'ตุลาคม': 'Q4', 'oct': 'Q4', 'october': 'Q4', '10': 'Q4', '10.0': 'Q4',
        'พ.ย.': 'Q4', 'พฤศจิกายน': 'Q4', 'nov': 'Q4', 'november': 'Q4', '11': 'Q4', '11.0': 'Q4',
        'ธ.ค.': 'Q4', 'ธันวาคม': 'Q4', 'dec': 'Q4', 'december': 'Q4', '12': 'Q4', '12.0': 'Q4',
    }

    # แปลงคอลัมน์เดือนเป็น string แล้วตัด space
    clean_month_series = df[month_col].astype(str).str.strip().str.lower()
    
    # Map เป็น Quarter
    df['quarter'] = clean_month_series.map(month_map)
    
    return df

# Apply ใช้งาน
df68 = extract_quarter_from_month(df68)
df69 = extract_quarter_from_month(df69)

print("df68 Quarter Counts:\n", df68['quarter'].value_counts(dropna=False))
print("\ndf69 Quarter Counts:\n", df69['quarter'].value_counts(dropna=False))

## 3. Merge Datasets & Feature Engineering

In [ ]:
# รวม Dataset
df = pd.concat([df68, df69], ignore_index=True)

# Clean String Fields
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "<NA>": np.nan})

# Cause / Campaign Theme Mapping
CAUSE_THEME_MAP = {
    "เบรกกะทันหัน": "Defensive Driving", "ตัดหน้า": "Defensive Driving", "รถตัดหน้า": "Defensive Driving",
    "ชนท้าย": "Defensive Driving", "ขับรถย้อนศร": "Defensive Driving", "เลี้ยวกระทันหัน": "Defensive Driving",
    "ใช้โทรศัพท์ขณะขับขี่": "Focus & Attention", "คุยโทรศัพท์": "Focus & Attention",
    "ผู้ขับขี่ผิดพลาดเอง(ตัดสินใจพลาด)": "Focus & Attention", "ง่วงนอน": "Focus & Attention", "ประมาท": "Focus & Attention",
    "ถนนลื่น": "Road & Vehicle Safety", "ถนนชำรุด": "Road & Vehicle Safety", "เสียหลักล้ม": "Road & Vehicle Safety",
    "ยางแตก": "Road & Vehicle Safety", "เบรกไม่อยู่": "Road & Vehicle Safety", "สัตว์ตัดหน้า": "Road & Vehicle Safety",
    "ขับรถเร็ว": "Speed Awareness", "ขับรถเร็วเกินกำหนด": "Speed Awareness", "เข้าโค้งเร็ว": "Speed Awareness"
}

cause_col = 'สาเหตุที่แท้จริงจากการเกิดอุบัติเหตุ (เช่น คุยโทรศัพท์ ขับรถมือเดียว รถตัดหน้า ซ้อนท้าย )'

def map_theme(val):
    if pd.isna(val): return "Defensive Driving"
    val_str = str(val).strip()
    for k, v in CAUSE_THEME_MAP.items():
        if k in val_str:
            return v
    return "Defensive Driving"

df['campaign_theme'] = df[cause_col].apply(map_theme)

# เติมค่า Missing ใน Feature สำคัญ
feature_cols = ['พื้นที่', 'ทัศนวิสัย', 'สภาพผิวจราจร', 'ลักษณะเส้นทาง', 'สภาพการจราจร', '4M1E']
for c in feature_cols:
    df[c] = df[c].fillna("Unknown")

print("Total Combined Records:", len(df))
print("Campaign Theme Distribution:\n", df['campaign_theme'].value_counts())

## 4. Train Model

In [ ]:
model_features = ['quarter', 'พื้นที่', 'ทัศนวิสัย', 'สภาพผิวจราจร', 'ลักษณะเส้นทาง', 'สภาพการจราจร', '4M1E']
X = df[model_features].copy()
y = df['campaign_theme'].copy()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = pd.DataFrame(encoder.fit_transform(X), columns=model_features)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_encoded, y)

print("Model Trained Successfully!")

## 5. Forecast Accident Probabilities (Q3 & Q4 / 2569) & 4M1E Mapping

In [ ]:
areas = df['พื้นที่'].unique()
quarters = ['Q3', 'Q4']

predict_rows = []
for q in quarters:
    for a in areas:
        area_df = df[df['พื้นที่'] == a]
        vis = area_df['ทัศนวิสัย'].mode()[0] if len(area_df) > 0 else "Unknown"
        surf = area_df['สภาพผิวจราจร'].mode()[0] if len(area_df) > 0 else "Unknown"
        road = area_df['ลักษณะเส้นทาง'].mode()[0] if len(area_df) > 0 else "Unknown"
        traf = area_df['สภาพการจราจร'].mode()[0] if len(area_df) > 0 else "Unknown"
        m4e = area_df['4M1E'].mode()[0] if len(area_df) > 0 else "Unknown"
        
        predict_rows.append({
            'quarter': q,
            'พื้นที่': a,
            'ทัศนวิสัย': vis,
            'สภาพผิวจราจร': surf,
            'ลักษณะเส้นทาง': road,
            'สภาพการจราจร': traf,
            '4M1E': m4e
        })

future_df = pd.DataFrame(predict_rows)
future_encoded = encoder.transform(future_df[model_features])

probs = rf_model.predict_proba(future_encoded)
classes = rf_model.classes_

for idx, c in enumerate(classes):
    future_df[f'Prob_{c} (%)'] = (probs[:, idx] * 100).round(2)

future_df['Predicted_Top_Theme'] = classes[np.argmax(probs, axis=1)]
future_df['Top_Theme_Probability (%)'] = np.max(probs, axis=1).round(4) * 100

future_df.head(10)

## 6. Summary Report & Export Predictions

In [ ]:
summary_cols = ['quarter', 'พื้นที่', 'Predicted_Top_Theme', 'Top_Theme_Probability (%)', '4M1E'] + [f'Prob_{c} (%)' for c in classes]
report_df = future_df[summary_cols].sort_values(by=['quarter', 'พื้นที่'])

report_df.to_csv("../data/forecast_q3_q4_2569_results.csv", index=False, encoding='utf-8-sig')
print("Forecast Results Exported Successfully!")
display(report_df)

---

แบบแยก sub-causes

In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# 1. โหลดข้อมูล
df68 = pd.read_excel("../data/300968-2568-AccidentRider-ฟอร์ม(ทวนสอบ130169)(2).xlsx", sheet_name=5)
df69 = pd.read_excel("../data/2569-AccidentRider-ฟอร์ม5.xlsx", sheet_name=0)

# 2. ลบ Unnamed และคอลัมน์ที่เป็นค่าว่างทั้งหมด
df68 = df68.loc[:, ~df68.columns.str.contains("^Unnamed", na=False)].dropna(axis=1, how="all")
df69 = df69.loc[:, ~df69.columns.str.contains("^Unnamed", na=False)].dropna(axis=1, how="all")

# 3. Align ชื่อคอลัมน์ให้ตรงกัน
rename_dict_69 = {
    "TradeArea จากร้านถึงจุดเกิดเหตุ": "Trade Area จากร้านถึงจุดเกิดเหตุ",
    "TradeArea ร้านถึงลูกค้ารับของ(กม.)": "Trade Area ร้านถึงลูกค้ารับของ (กม.)",
    "ผลของการเกิดอุบัติเหตุ (เสียชีวิต/สูญเสียอวัยวะ/บาดเจ็บ)": "(เสียชีวิต/สูญเสียอวัยวะ/บาดเจ็บ)"
}
df69 = df69.rename(columns=rename_dict_69)
df69 = df69.drop(columns=["หลักสูตรความปลอดภัย6ชม.(ไม่เก็บเงิน)"], errors="ignore")

print("df68 Shape:", df68.shape)
print("df69 Shape:", df69.shape)

In [ ]:
def extract_quarter_from_month(df):
    # ค้นหาคอลัมน์เดือน
    month_col = None
    for col in df.columns:
        if "เดือน" in str(col):
            month_col = col
            break
            
    if month_col is None:
        print("⚠️ ไม่พบคอลัมน์เดือนใน DataFrame")
        return df

    month_map = {
        # Q1
        'ม.ค.': 'Q1', 'มกราคม': 'Q1', 'jan': 'Q1', 'january': 'Q1', '1': 'Q1', '01': 'Q1', '1.0': 'Q1',
        'ก.พ.': 'Q1', 'กุมภาพันธ์': 'Q1', 'feb': 'Q1', 'february': 'Q1', '2': 'Q1', '02': 'Q1', '2.0': 'Q1',
        'มี.ค.': 'Q1', 'มีนาคม': 'Q1', 'mar': 'Q1', 'march': 'Q1', '3': 'Q1', '03': 'Q1', '3.0': 'Q1',
        # Q2
        'เม.ย.': 'Q2', 'เมษายน': 'Q2', 'apr': 'Q2', 'april': 'Q2', '4': 'Q2', '04': 'Q2', '4.0': 'Q2',
        'พ.ค.': 'Q2', 'พฤษภาคม': 'Q2', 'may': 'Q2', '5': 'Q2', '05': 'Q2', '5.0': 'Q2',
        'มิ.ย.': 'Q2', 'มิถุนายน': 'Q2', 'jun': 'Q2', 'june': 'Q2', '6': 'Q2', '06': 'Q2', '6.0': 'Q2',
        # Q3
        'ก.ค.': 'Q3', 'กรกฎาคม': 'Q3', 'jul': 'Q3', 'july': 'Q3', '7': 'Q3', '07': 'Q3', '7.0': 'Q3',
        'ส.ค.': 'Q3', 'สิงหาคม': 'Q3', 'aug': 'Q3', 'august': 'Q3', '8': 'Q3', '08': 'Q3', '8.0': 'Q3',
        'ก.ย.': 'Q3', 'กันยายน': 'Q3', 'sep': 'Q3', 'september': 'Q3', '9': 'Q3', '09': 'Q3', '9.0': 'Q3',
        # Q4
        'ต.ค.': 'Q4', 'ตุลาคม': 'Q4', 'oct': 'Q4', 'october': 'Q4', '10': 'Q4', '10.0': 'Q4',
        'พ.ย.': 'Q4', 'พฤศจิกายน': 'Q4', 'nov': 'Q4', 'november': 'Q4', '11': 'Q4', '11.0': 'Q4',
        'ธ.ค.': 'Q4', 'ธันวาคม': 'Q4', 'dec': 'Q4', 'december': 'Q4', '12': 'Q4', '12.0': 'Q4',
    }

    clean_month_series = df[month_col].astype(str).str.strip().str.lower()
    df['quarter'] = clean_month_series.map(month_map)
    return df

df68 = extract_quarter_from_month(df68)
df69 = extract_quarter_from_month(df69)

print("df68 Quarter Counts:\n", df68['quarter'].value_counts(dropna=False))
print("\ndf69 Quarter Counts:\n", df69['quarter'].value_counts(dropna=False))

In [ ]:
# 1. รวม Dataset
df = pd.concat([df68, df69], ignore_index=True)

# Clean String Fields
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "<NA>": np.nan})

# 2. Map สาเหตุ -> 4 Themes หลัก
CAUSE_THEME = {
    # Defensive Driving
    "เบรกกะทันหัน": "Defensive Driving", "ขับรถย้อนศร": "Defensive Driving", "ซ้อนท้าย": "Defensive Driving",
    "เลี้ยวกระทันหัน": "Defensive Driving", "ฝ่าสัญญาณไฟจราจร": "Defensive Driving", "ตัดหน้า": "Defensive Driving",
    "รถตัดหน้า": "Defensive Driving", "คนเดินข้ามถนนตัดหน้ารถ": "Defensive Driving", "ชนท้าย": "Defensive Driving",
    "เปลี่ยนช่องทางกระทันหัน": "Defensive Driving", "เฉี่ยวชนคู่กรณี": "Defensive Driving", "แซงไม่พ้น": "Defensive Driving",
    # Focus & Attention
    "ใช้โทรศัพท์ขณะขับขี่": "Focus & Attention", "คุยโทรศัพท์": "Focus & Attention", "ผู้ขับขี่ผิดพลาดเอง(ตัดสินใจพลาด)": "Focus & Attention",
    "ง่วงนอน": "Focus & Attention", "ขับรถหลับใน": "Focus & Attention", "ประมาท": "Focus & Attention",
    "มองไม่เห็น": "Focus & Attention", "ไม่ดูกระจกมองข้าง": "Focus & Attention", "ไม่คุ้นชินเส้นทาง": "Focus & Attention",
    # Road & Vehicle Safety
    "เสียหลักล้ม": "Road & Vehicle Safety", "สัตว์ตัดหน้า": "Road & Vehicle Safety", "ถนนชำรุด": "Road & Vehicle Safety",
    "ถนนลื่น": "Road & Vehicle Safety", "ฝนตกถนนลื่น": "Road & Vehicle Safety", "หลุมบ่อ": "Road & Vehicle Safety",
    "เบรกไม่อยู่": "Road & Vehicle Safety", "ยางแตก": "Road & Vehicle Safety", "โซ่หลุด": "Road & Vehicle Safety",
    # Speed Awareness
    "ขับรถเร็ว": "Speed Awareness", "ขับรถเร็วเกินกำหนด": "Speed Awareness", "เข้าโค้งเร็ว": "Speed Awareness", "ออกตัวเร็ว": "Speed Awareness"
}

cause_col = 'สาเหตุที่แท้จริงจากการเกิดอุบัติเหตุ (เช่น คุยโทรศัพท์ ขับรถมือเดียว รถตัดหน้า ซ้อนท้าย )'

def map_theme(val):
    if pd.isna(val): return "Defensive Driving"
    val_str = str(val).strip()
    for k, v in CAUSE_THEME.items():
        if k in val_str:
            return v
    return "Defensive Driving"

df['campaign_theme'] = df[cause_col].apply(map_theme)

# 3. Classify 4M1E อัตโนมัติจากสาเหตุจริง
def classify_4m1e(row):
    cause = str(row.get(cause_col, '')).lower()
    surface = str(row.get('สภาพผิวจราจร', '')).lower()
    
    if any(k in cause for k in ['ถนน', 'ลื่น', 'ชำรุด', 'สัตว์', 'หลุม', 'ฝน']) or 'ลื่น' in surface:
        return 'Environment'
    if any(k in cause for k in ['ยางแตก', 'เบรกไม่อยู่', 'เบรกขัดข้อง', 'รถเสีย', 'โซ่หลุด']):
        return 'Machine'
    if any(k in cause for k in ['ย้อนศร', 'ผิดกฎ', 'แซง']):
        return 'Method'
    return 'Man'

df['4M1E_Cleaned'] = df.apply(classify_4m1e, axis=1)

# เติม Missing ค่าใน Feature หลัก
feature_cols = ['พื้นที่', 'ทัศนวิสัย', 'สภาพผิวจราจร', 'ลักษณะเส้นทาง', 'สภาพการจราจร', '4M1E_Cleaned']
for c in feature_cols:
    df[c] = df[c].fillna("Unknown")

print("Combined Data Shape:", df.shape)
print("Campaign Theme Distribution:\n", df['campaign_theme'].value_counts())

In [ ]:
# ------------------------------------------------------------------
# 1. PREPARE DATA & TRAIN-TEST SPLIT
# ------------------------------------------------------------------
model_features = ['quarter', 'พื้นที่', 'ทัศนวิสัย', 'สภาพผิวจราจร', 'ลักษณะเส้นทาง', 'สภาพการจราจร', '4M1E_Cleaned']
X = df[model_features].copy()
y = df['campaign_theme'].copy()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = pd.DataFrame(encoder.fit_transform(X), columns=model_features)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Train Model สำหรับ Evaluation
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# ------------------------------------------------------------------
# 2. ACCURACY & FEATURE IMPORTANCE
# ------------------------------------------------------------------
y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("="*60)
print(f"📊 MODEL ACCURACY: {acc * 100:.2f}%")
print("="*60)
print("\n📋 CLASSIFICATION REPORT:\n")
print(classification_report(y_test, y_pred))

importances = rf_model.feature_importances_
feature_imp_df = pd.DataFrame({
    'Feature': model_features,
    'Importance (%)': (importances * 100).round(2)
}).sort_values(by='Importance (%)', ascending=False)

print("="*60)
print("🔍 FEATURE IMPORTANCE:")
print("="*60)
print(feature_imp_df.to_string(index=False))

# ------------------------------------------------------------------
# 3. TWO-STAGE FORECASTING (Q3 - Q4 / 2569)
# ------------------------------------------------------------------
# Retrain บน Dataset ทั้งหมด
rf_model.fit(X_encoded, y)

areas = df['พื้นที่'].unique()
quarters = ['Q3', 'Q4']
predict_rows = []

for q in quarters:
    for a in areas:
        area_df = df[df['พื้นที่'] == a]
        if len(area_df) == 0: continue
        
        predict_rows.append({
            'quarter': q,
            'พื้นที่': a,
            'ทัศนวิสัย': area_df['ทัศนวิสัย'].mode()[0],
            'สภาพผิวจราจร': area_df['สภาพผิวจราจร'].mode()[0],
            'ลักษณะเส้นทาง': area_df['ลักษณะเส้นทาง'].mode()[0],
            'สภาพการจราจร': area_df['สภาพการจราจร'].mode()[0],
            '4M1E_Cleaned': area_df['4M1E_Cleaned'].mode()[0]
        })

future_df = pd.DataFrame(predict_rows)
future_encoded = encoder.transform(future_df[model_features])

# Stage 1: Predict Theme Probabilities
probs = rf_model.predict_proba(future_encoded)
classes = rf_model.classes_

# Stage 2: Breakdown into Sub-Causes
forecast_results = []

for idx, row in future_df.iterrows():
    area = row['พื้นที่']
    q = row['quarter']
    
    for c_idx, theme in enumerate(classes):
        theme_prob = probs[idx, c_idx]
        area_causes = df[(df['พื้นที่'] == area) & (df['campaign_theme'] == theme)]
        
        if len(area_causes) > 0:
            top_causes = area_causes[cause_col].value_counts(normalize=True).head(3)
            
            for cause_name, ratio in top_causes.items():
                estimated_cause_prob = theme_prob * ratio * 100
                
                forecast_results.append({
                    'Quarter': q,
                    'พื้นที่': area,
                    'Predicted_Theme': theme,
                    'Theme_Probability (%)': round(theme_prob * 100, 2),
                    'Sub_Cause (สาเหตุย่อย)': cause_name,
                    'Historical_Ratio_in_Theme (%)': round(ratio * 100, 2),
                    'Estimated_Cause_Probability (%)': round(estimated_cause_prob, 2)
                })

final_forecast_df = pd.DataFrame(forecast_results)

# Export เป็น CSV
final_forecast_df.to_csv("../data/forecast_sub_causes_q3_q4_2569.csv", index=False, encoding='utf-8-sig')

print("\n="*60)
print("🔮 FORECAST RESULTS SAMPLE (Top 10):")
print("="*60)
display(final_forecast_df.sort_values(by=['Quarter', 'พื้นที่', 'Estimated_Cause_Probability (%)'], ascending=[True, True, False]).head(10))

---
แยก sub-causes **แบบรวม code** ใน block เดียว (เอาไปใส่ให้รันจาก data อัตโนมัติ ไม่ต้องรอเอาผลมันใส่เอง ตอนแสดงผลหน้า web)

In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# ==============================================================================
# 1. LOAD DATA & CLEAN COLUMNS
# ==============================================================================
# โหลดข้อมูล
df68 = pd.read_excel("../data/300968-2568-AccidentRider-ฟอร์ม(ทวนสอบ130169)(2).xlsx", sheet_name=5)
df69 = pd.read_excel("../data/2569-AccidentRider-ฟอร์ม5.xlsx", sheet_name=0)

# ลบ Unnamed และคอลัมน์ที่เป็นค่าว่างทั้งหมด
df68 = df68.loc[:, ~df68.columns.str.contains("^Unnamed", na=False)].dropna(axis=1, how="all")
df69 = df69.loc[:, ~df69.columns.str.contains("^Unnamed", na=False)].dropna(axis=1, how="all")

# Align ชื่อคอลัมน์ให้ตรงกัน
rename_dict_69 = {
    "TradeArea จากร้านถึงจุดเกิดเหตุ": "Trade Area จากร้านถึงจุดเกิดเหตุ",
    "TradeArea ร้านถึงลูกค้ารับของ(กม.)": "Trade Area ร้านถึงลูกค้ารับของ (กม.)",
    "ผลของการเกิดอุบัติเหตุ (เสียชีวิต/สูญเสียอวัยวะ/บาดเจ็บ)": "(เสียชีวิต/สูญเสียอวัยวะ/บาดเจ็บ)"
}
df69 = df69.rename(columns=rename_dict_69)
df69 = df69.drop(columns=["หลักสูตรความปลอดภัย6ชม.(ไม่เก็บเงิน)"], errors="ignore")

# เพิ่มคอลัมน์ ระบุปี พ.ศ. ชัดเจน
df68['year'] = 2568
df69['year'] = 2569

print("df68 Shape:", df68.shape)
print("df69 Shape:", df69.shape)

# ==============================================================================
# 2. EXTRACT QUARTER FROM MONTH
# ==============================================================================
def extract_quarter_from_month(df):
    month_col = None
    for col in df.columns:
        if "เดือน" in str(col):
            month_col = col
            break
            
    if month_col is None:
        print("⚠️ ไม่พบคอลัมน์เดือนใน DataFrame")
        return df

    month_map = {
        'ม.ค.': 'Q1', 'มกราคม': 'Q1', 'jan': 'Q1', 'january': 'Q1', '1': 'Q1', '01': 'Q1', '1.0': 'Q1',
        'ก.พ.': 'Q1', 'กุมภาพันธ์': 'Q1', 'feb': 'Q1', 'february': 'Q1', '2': 'Q1', '02': 'Q1', '2.0': 'Q1',
        'มี.ค.': 'Q1', 'มีนาคม': 'Q1', 'mar': 'Q1', 'march': 'Q1', '3': 'Q1', '03': 'Q1', '3.0': 'Q1',
        'เม.ย.': 'Q2', 'เมษายน': 'Q2', 'apr': 'Q2', 'april': 'Q2', '4': 'Q2', '04': 'Q2', '4.0': 'Q2',
        'พ.ค.': 'Q2', 'พฤษภาคม': 'Q2', 'may': 'Q2', '5': 'Q2', '05': 'Q2', '5.0': 'Q2',
        'มิ.ย.': 'Q2', 'มิถุนายน': 'Q2', 'jun': 'Q2', 'june': 'Q2', '6': 'Q2', '06': 'Q2', '6.0': 'Q2',
        'ก.ค.': 'Q3', 'กรกฎาคม': 'Q3', 'jul': 'Q3', 'july': 'Q3', '7': 'Q3', '07': 'Q3', '7.0': 'Q3',
        'ส.ค.': 'Q3', 'สิงหาคม': 'Q3', 'aug': 'Q3', 'august': 'Q3', '8': 'Q3', '08': 'Q3', '8.0': 'Q3',
        'ก.ย.': 'Q3', 'กันยายน': 'Q3', 'sep': 'Q3', 'september': 'Q3', '9': 'Q3', '09': 'Q3', '9.0': 'Q3',
        'ต.ค.': 'Q4', 'ตุลาคม': 'Q4', 'oct': 'Q4', 'october': 'Q4', '10': 'Q4', '10.0': 'Q4',
        'พ.ย.': 'Q4', 'พฤศจิกายน': 'Q4', 'nov': 'Q4', 'november': 'Q4', '11': 'Q4', '11.0': 'Q4',
        'ธ.ค.': 'Q4', 'ธันวาคม': 'Q4', 'dec': 'Q4', 'december': 'Q4', '12': 'Q4', '12.0': 'Q4',
    }

    clean_month_series = df[month_col].astype(str).str.strip().str.lower()
    df['quarter'] = clean_month_series.map(month_map)
    return df

df68 = extract_quarter_from_month(df68)
df69 = extract_quarter_from_month(df69)

# ==============================================================================
# 3. MERGE, THEME MAPPING & 4M1E CLASSIFICATION
# ==============================================================================
df = pd.concat([df68, df69], ignore_index=True)

# Clean String Fields
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "<NA>": np.nan})

# Map สาเหตุ -> 4 Themes หลัก
CAUSE_THEME = {
    "เบรกกะทันหัน": "Defensive Driving", "ขับรถย้อนศร": "Defensive Driving", "ซ้อนท้าย": "Defensive Driving",
    "เลี้ยวกระทันหัน": "Defensive Driving", "ฝ่าสัญญาณไฟจราจร": "Defensive Driving", "ตัดหน้า": "Defensive Driving",
    "รถตัดหน้า": "Defensive Driving", "คนเดินข้ามถนนตัดหน้ารถ": "Defensive Driving", "ชนท้าย": "Defensive Driving",
    "เปลี่ยนช่องทางกระทันหัน": "Defensive Driving", "เฉี่ยวชนคู่กรณี": "Defensive Driving", "แซงไม่พ้น": "Defensive Driving",
    "ใช้โทรศัพท์ขณะขับขี่": "Focus & Attention", "คุยโทรศัพท์": "Focus & Attention", "ผู้ขับขี่ผิดพลาดเอง(ตัดสินใจพลาด)": "Focus & Attention",
    "ง่วงนอน": "Focus & Attention", "ขับรถหลับใน": "Focus & Attention", "ประมาท": "Focus & Attention",
    "มองไม่เห็น": "Focus & Attention", "ไม่ดูกระจกมองข้าง": "Focus & Attention", "ไม่คุ้นชินเส้นทาง": "Focus & Attention",
    "เสียหลักล้ม": "Road & Vehicle Safety", "สัตว์ตัดหน้า": "Road & Vehicle Safety", "ถนนชำรุด": "Road & Vehicle Safety",
    "ถนนลื่น": "Road & Vehicle Safety", "ฝนตกถนนลื่น": "Road & Vehicle Safety", "หลุมบ่อ": "Road & Vehicle Safety",
    "เบรกไม่อยู่": "Road & Vehicle Safety", "ยางแตก": "Road & Vehicle Safety", "โซ่หลุด": "Road & Vehicle Safety",
    "ขับรถเร็ว": "Speed Awareness", "ขับรถเร็วเกินกำหนด": "Speed Awareness", "เข้าโค้งเร็ว": "Speed Awareness", "ออกตัวเร็ว": "Speed Awareness"
}

cause_col = 'สาเหตุที่แท้จริงจากการเกิดอุบัติเหตุ (เช่น คุยโทรศัพท์ ขับรถมือเดียว รถตัดหน้า ซ้อนท้าย )'

def map_theme(val):
    if pd.isna(val): return "Defensive Driving"
    val_str = str(val).strip()
    for k, v in CAUSE_THEME.items():
        if k in val_str:
            return v
    return "Defensive Driving"

df['campaign_theme'] = df[cause_col].apply(map_theme)

# Classify 4M1E อัตโนมัติจากสาเหตุจริง
def classify_4m1e(row):
    cause = str(row.get(cause_col, '')).lower()
    surface = str(row.get('สภาพผิวจราจร', '')).lower()
    
    if any(k in cause for k in ['ถนน', 'ลื่น', 'ชำรุด', 'สัตว์', 'หลุม', 'ฝน']) or 'ลื่น' in surface:
        return 'Environment'
    if any(k in cause for k in ['ยางแตก', 'เบรกไม่อยู่', 'เบรกขัดข้อง', 'รถเสีย', 'โซ่หลุด']):
        return 'Machine'
    if any(k in cause for k in ['ย้อนศร', 'ผิดกฎ', 'แซง']):
        return 'Method'
    return 'Man'

df['4M1E_Cleaned'] = df.apply(classify_4m1e, axis=1)

# เติม Missing ค่าใน Feature หลัก
feature_cols = ['พื้นที่', 'ทัศนวิสัย', 'สภาพผิวจราจร', 'ลักษณะเส้นทาง', 'สภาพการจราจร', '4M1E_Cleaned']
for c in feature_cols:
    df[c] = df[c].fillna("Unknown")

print("\nCombined Data Shape:", df.shape)

# ==============================================================================
# 4. TRAIN MODEL & EVALUATION
# ==============================================================================
model_features = ['quarter', 'พื้นที่', 'ทัศนวิสัย', 'สภาพผิวจราจร', 'ลักษณะเส้นทาง', 'สภาพการจราจร', '4M1E_Cleaned']
X = df[model_features].copy()
y = df['campaign_theme'].copy()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = pd.DataFrame(encoder.fit_transform(X), columns=model_features)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("\n" + "="*60)
print(f"📊 MODEL ACCURACY: {acc * 100:.2f}%")
print("="*60)
print("\n📋 CLASSIFICATION REPORT:\n")
print(classification_report(y_test, y_pred))

importances = rf_model.feature_importances_
feature_imp_df = pd.DataFrame({
    'Feature': model_features,
    'Importance (%)': (importances * 100).round(2)
}).sort_values(by='Importance (%)', ascending=False)

print("="*60)
print("🔍 FEATURE IMPORTANCE:")
print("="*60)
print(feature_imp_df.to_string(index=False))

# ==============================================================================
# 5. TWO-STAGE FORECASTING WITH Q1-Q2 YOY TREND WEIGHTING (Q3 & Q4 / 2569)
# ==============================================================================
# Retrain บนข้อมูลทั้งหมด
rf_model.fit(X_encoded, y)

areas = df['พื้นที่'].unique()
quarters = ['Q3', 'Q4']
predict_rows = []

for q in quarters:
    for a in areas:
        area_df = df[df['พื้นที่'] == a]
        if len(area_df) == 0: continue
        
        predict_rows.append({
            'quarter': q,
            'พื้นที่': a,
            'ทัศนวิสัย': area_df['ทัศนวิสัย'].mode()[0],
            'สภาพผิวจราจร': area_df['สภาพผิวจราจร'].mode()[0],
            'ลักษณะเส้นทาง': area_df['ลักษณะเส้นทาง'].mode()[0],
            'สภาพการจราจร': area_df['สภาพการจราจร'].mode()[0],
            '4M1E_Cleaned': area_df['4M1E_Cleaned'].mode()[0]
        })

future_df = pd.DataFrame(predict_rows)
future_encoded = encoder.transform(future_df[model_features])

# Stage 1: Predict Theme Probabilities
probs = rf_model.predict_proba(future_encoded)
classes = rf_model.classes_

# Stage 2: Breakdown into Sub-Causes + Q1-Q2 YoY Trend Multiplier
forecast_results = []

for idx, row in future_df.iterrows():
    area = row['พื้นที่']
    q = row['quarter']
    
    for c_idx, theme in enumerate(classes):
        theme_prob = probs[idx, c_idx]
        area_df = df[df['พื้นที่'] == area]
        
        # คำนวณ Trend Shift (เทียบ Q1-Q2 ปี 68 vs Q1-Q2 ปี 69)
        q12_68 = area_df[(area_df['year'] == 2568) & (area_df['quarter'].isin(['Q1', 'Q2'])) & (area_df['campaign_theme'] == theme)]
        q12_69 = area_df[(area_df['year'] == 2569) & (area_df['quarter'].isin(['Q1', 'Q2'])) & (area_df['campaign_theme'] == theme)]
        
        total_68 = len(area_df[area_df['year'] == 2568])
        total_69 = len(area_df[area_df['year'] == 2569])
        
        trend_factor = 1.0
        if total_68 > 0 and total_69 > 0:
            ratio_68 = len(q12_68) / total_68
            ratio_69 = len(q12_69) / total_69
            if ratio_68 > 0:
                trend_factor = ratio_69 / ratio_68
                trend_factor = np.clip(trend_factor, 0.8, 1.5) # Cap ช่วงน้ำหนักไว้ที่ 0.8 - 1.5

        area_causes = area_df[area_df['campaign_theme'] == theme]
        if len(area_causes) > 0:
            top_causes = area_causes[cause_col].value_counts(normalize=True).head(3)
            
            for cause_name, ratio in top_causes.items():
                adjusted_cause_prob = theme_prob * ratio * trend_factor * 100
                
                forecast_results.append({
                    'Quarter': q,
                    'พื้นที่': area,
                    'Predicted_Theme': theme,
                    'Base_Theme_Prob (%)': round(theme_prob * 100, 2),
                    'Q1-Q2_Trend_Multiplier': round(trend_factor, 2),
                    'Sub_Cause (สาเหตุย่อย)': cause_name,
                    'Estimated_Cause_Probability (%)': round(adjusted_cause_prob, 2)
                })

final_forecast_df = pd.DataFrame(forecast_results)

# Export ผลลัพธ์ออกเป็น CSV
final_forecast_df.to_csv("../data/forecast_sub_causes_q3_q4_2569.csv", index=False, encoding='utf-8-sig')

# ==============================================================================
# 6. DISPLAY RESULTS SAMPLE
# ==============================================================================
print("\n" + "="*60)
print("🔮 FORECAST RESULTS SAMPLE - Q3 (Top 13):")
print("="*60)
display(final_forecast_df[final_forecast_df['Quarter'] == 'Q3'].sort_values(by=['Estimated_Cause_Probability (%)'], ascending=False).head(13))

print("\n" + "="*60)
print("🔮 FORECAST RESULTS SAMPLE - Q4 (Top 14):")
print("="*60)
display(final_forecast_df[final_forecast_df['Quarter'] == 'Q4'].sort_values(by=['Estimated_Cause_Probability (%)'], ascending=False).head(14))

In [ ]:
# ดูการกระจายตัวของจำนวนเคสพยากรณ์แยกตาม Quarter
print("จำนวนเคสพยากรณ์แยกตามไตรมาส:")
print(final_forecast_df['Quarter'].value_counts())